In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine,text
import urllib.parse
from sqlalchemy.types import NVARCHAR

In [2]:
folder = Path("../data/raw/mandat_uzbmb_uz_2025")
data =  pd.concat((pd.read_csv(file,encoding='utf-8-sig') for file in folder.glob("*.csv")),ignore_index=True)

In [3]:
data["TR class"] = data["TR class"].apply(lambda x: 'Grant' if x == 'table-success'else 'Kontrakt' if x == 'table-warning' else 'Yiqilgan')

In [4]:
data["TR class"].unique() 

array(['Yiqilgan', 'Grant', 'Kontrakt'], dtype=object)

In [5]:
data.columns

Index(['ID', 'F.I.SH', 'Ball', 'Yo'nalish', 'Oliy ta'lim muassasasi',
       'Ta'lim tili', 'Ta'lim shakli', 'Batafsil', 'TR class'],
      dtype='object')

In [6]:
data.rename(columns={'F.I.SH':'Ism_Sharif',"Yo'nalish":"Mutahasislik","Oliy ta'lim muassasasi":"Universitet","Ta'lim tili":"Til","Ta'lim shakli":"Turi","TR class":"Natija"},inplace=True)

In [7]:
data.dtypes

ID              object
Ism_Sharif      object
Ball            object
Mutahasislik    object
Universitet     object
Til             object
Turi            object
Batafsil        object
Natija          object
dtype: object

In [8]:
data['Ball'] = pd.to_numeric(data['Ball'].astype(str).str.replace(',', '.'), errors='coerce')
    

In [9]:
data['ID'] = pd.to_numeric(data['ID'], errors='coerce') # converting obj to int

In [10]:
data['ID'].dtype

dtype('int64')

In [11]:
data.head()

,ID,Ism_Sharif,Ball,Mutahasislik,Universitet,Til,Turi,Batafsil,Natija
0,6060954,TURDIMURODOVA F. N.,172.0,"Milliy gʻoya, maʼnaviyat asoslari va huquq taʼ...",Andijon davlat universiteti,O'zbekcha,Kunduzgi,Batafsil,Yiqilgan
1,5523352,BAXOROV E. G‘.,169.6,"Milliy gʻoya, maʼnaviyat asoslari va huquq taʼ...",Andijon davlat universiteti,O'zbekcha,Kunduzgi,Batafsil,Yiqilgan
2,5524834,QURBANBAYEVA R. E.,168.6,"Milliy gʻoya, maʼnaviyat asoslari va huquq taʼ...",Andijon davlat universiteti,O'zbekcha,Kunduzgi,Batafsil,Yiqilgan
3,5510736,JALILOVA R. R.,166.4,"Milliy gʻoya, maʼnaviyat asoslari va huquq taʼ...",Andijon davlat universiteti,O'zbekcha,Kunduzgi,Batafsil,Grant
4,6045821,SHAMURATOV N. U.,164.1,"Milliy gʻoya, maʼnaviyat asoslari va huquq taʼ...",Andijon davlat universiteti,O'zbekcha,Kunduzgi,Batafsil,Yiqilgan


In [12]:
data['Batafsil'].nunique()

1

In [13]:
data.drop(columns='Batafsil',inplace=True)

In [14]:
data[data['Ball'].isna()]

,ID,Ism_Sharif,Ball,Mutahasislik,Universitet,Til,Turi,Natija


In [15]:
data.dropna(subset='Ball',inplace=True)

In [16]:
data.isna().sum()

ID              0
Ism_Sharif      0
Ball            0
Mutahasislik    0
Universitet     0
Til             0
Turi            0
Natija          0
dtype: int64

In [17]:
data.duplicated().sum()

np.int64(0)

In [18]:

data.to_csv("../data/cleaned/Cleaned_Mandat.csv", index=False,encoding='utf-8')

In [19]:
# # this code is for loadind data to existing tbl in ssms

params = urllib.parse.quote_plus(
    #dont miss ; and pay attention to any Letter case
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Mandat_Database;"
    "Trusted_Connection=yes;"
)

# 4. Create Engine (for million rows fast_executemany)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)

with engine.begin() as conn:
    conn.execute(text("""
    IF OBJECT_ID('dbo.Mandat', 'U') IS NOT NULL
        DROP TABLE dbo.Mandat;

    CREATE TABLE dbo.Mandat (
        ID INT NULL,
        Ism_Sharif NVARCHAR(255),
        Mutahasislik NVARCHAR(255),
        Universitet NVARCHAR(255),
        Til NVARCHAR(50),
        Turi NVARCHAR(50),
        Natija NVARCHAR(50),
        Ball FLOAT
    );
    """))

 # Avoiding encoding problem 
dtype_map ={col:NVARCHAR(255) for col in data.select_dtypes(include=['object']).columns}

# upload data to SQL Server
try:
    print(f"Uploading has started: {len(data)} rows of data...")
    # chunksize=30000 -> Reducing RAM usage and improving performance for large datasets
    data.to_sql('Mandat', con=engine, if_exists='append', index=False,dtype=dtype_map, chunksize=30000)
    print("✅ All data has been uploaded successfully!")
except Exception as e:
    print(f"❌ Error: {e}")

Uploading has started: 1088341 rows of data...
✅ All data has been uploaded successfully!
